# 04 — V-Order

## Story beat: "Pay a little on write, save on every read"

Contoso's gold tables are read constantly by Power BI and the warehouse. **V-Order** is a write-time Parquet layout (sort + dictionary encoding + row-group tuning) that makes those reads faster.

**Key facts (Microsoft docs):**
- **Verti-Scan engines (Power BI, SQL) get the big win** — near in-memory read times.
- **Direct Lake mode *depends* on V-Order.**
- **Spark** reads benefit more modestly — ~10% on average.
- V-Order is **off by default** in new Fabric workspaces (write-heavy profile); the **warehouse applies it automatically**.

This notebook writes the same data with V-Order off vs on and compares file size. The headline *read* benefit is realized in **Module 4 (Direct Lake)** and the warehouse — not in a Spark stopwatch, so treat the timing here as indicative only.

In [ ]:
%run 00_config

## Step 1 — Build a larger demo fact

V-Order is meaningless on ~1 MB in a single file, so we scale `silver.fact_sales` up to ~2M rows — with a tiny jitter so rows are realistic rather than exact duplicates. That gives multiple files / row groups where the layout actually matters.

In [ ]:
import time
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS demo")
src = (
    spark.table(f"{SILVER_SCHEMA}.fact_sales")
    .crossJoin(spark.range(40))                                  # ~2M rows
    .withColumn("net_amount", F.round(F.col("net_amount") * (1 + F.rand() * 0.001), 4))  # jitter -> distinct
    .drop("id")
    .repartition(8)
)
print("demo rows:", src.count())

## Step 2 — Write the same data, V-Order off vs on

| Table | V-Order |
| --- | --- |
| `demo.fact_no_vorder` | Off (session default) |
| `demo.fact_vorder` | On — `OPTIMIZE ... VORDER` forces the layout regardless of session |

In [ ]:
# Baseline: V-Order OFF (the write-heavy default in new workspaces).
spark.conf.set("spark.sql.parquet.vorder.default", "false")
src.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("demo.fact_no_vorder")

# V-Order ON: write, then OPTIMIZE ... VORDER rewrites the files in V-Order layout (reliable, doc-recommended).
src.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("demo.fact_vorder")
spark.sql("ALTER TABLE demo.fact_vorder SET TBLPROPERTIES ('delta.parquet.vorder.enabled' = 'true')")
spark.sql("OPTIMIZE demo.fact_vorder VORDER")
print("both tables written")

## Step 3 — Compare on-disk size

`DESCRIBE DETAIL` returns total size + file count. V-Order can reduce bytes through better sorting/encoding; the effect is **data-dependent and modest on synthetic data**, so don't expect a dramatic gap here.

In [ ]:
def detail(name):
    d = spark.sql(f"DESCRIBE DETAIL {name}").collect()[0]
    return d["sizeInBytes"], d["numFiles"]

for t in ["demo.fact_no_vorder", "demo.fact_vorder"]:
    size, files = detail(t)
    print(f"{t:24s} size={size/1024/1024:8.2f} MB  files={files}")

# Confirm V-Order is actually enabled on the second table.
props = spark.sql("SHOW TBLPROPERTIES demo.fact_vorder").toPandas()
print("\nV-Order property:")
print(props[props['key'].str.contains('vorder', case=False)].to_string(index=False))

## Step 4 — Read time (indicative only)

A filtered aggregation, averaged over several runs. Expect only a **small** Spark difference — that is by design: Spark is not a Verti-Scan engine. The real V-Order payoff is the read path in **Power BI / SQL** and **Direct Lake (Module 4, which requires V-Order)**.

In [ ]:
def avg_time(name, runs=5):
    q = f"SELECT store_id, SUM(net_amount) FROM {name} WHERE store_id IN (3,7,9) GROUP BY store_id"
    spark.sql(q).collect()  # warm
    ts = []
    for _ in range(runs):
        t0 = time.time(); spark.sql(q).collect(); ts.append(time.time() - t0)
    return round(sum(ts) / len(ts), 3)

print("no V-Order avg:", avg_time("demo.fact_no_vorder"), "s")
print("V-Order    avg:", avg_time("demo.fact_vorder"), "s")
print("\nSpark gains from V-Order are modest (~10% avg per Microsoft docs).")
print("The large win is Power BI / SQL Verti-Scan + Direct Lake (which requires V-Order) - see Module 4.")

---

## ✅ What this shows

- Same data written V-Order **off vs on**, with size + file metadata compared and the V-Order property confirmed.
- The Spark read difference is **modest by design** (~10%) — don't oversell it.
- The real benefit lands in **Module 4 (Direct Lake — requires V-Order)** and the **warehouse** (V-Order on by default).

**Next:** Module 2 — the warehouse reads the same gold tables in T-SQL.